# Geometría de los embeddings: ¿la silueta baja se debe al espacio o a que falta información?

Con los embeddings crudos, la silueta (distancia coseno) del OE3 sale cerca de 0. Los embeddings de BERT y modelos parecidos tienen dos problemas conocidos que bajan esa métrica aunque la información de la intención esté: todos los vectores se concentran en un cono (anisotropía) y unas pocas dimensiones dominan las distancias.

Este cuaderno extrae los embeddings una sola vez, les aplica siete transformaciones que no usan las etiquetas y recalcula silueta, Davies-Bouldin y Calinski-Harabasz. Además:

- calcula la silueta que daría un reparto al azar de las intenciones (50 permutaciones), para saber qué significa "cerca de 0" en cada espacio;
- calcula las mismas métricas con el vector TF-IDF de n-gramas de letras, como control sin modelo de lenguaje.

Las transformaciones están en `transformaciones_embeddings.py`, se definieron antes de ver los resultados y se reportan todas, mejoren o no la silueta. No usa la API ni entrena modelos. En una GPU T4 toma entre 10 y 15 minutos; los embeddings quedan guardados en Drive.

## 0. Montar Google Drive (aquí quedan los embeddings y los resultados)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repositorio

Antes de correrlo, haz `git push` de `transformaciones_embeddings.py` (esta celda lo verifica).

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git log --oneline -3
!pip uninstall -y -q torchao

REPO = '/content/shiwilu-tesis'
requeridos = [
    f'{REPO}/corpus/corpus_shiwilu_final.csv',
    f'{REPO}/oe2_aumento_de_datos/particiones/corpus_normalizado.csv',
    f'{REPO}/oe3_caracterizacion_embeddings/caracterizacion.py',
    f'{REPO}/oe3_caracterizacion_embeddings/transformaciones_embeddings.py',
    f'{REPO}/oe3_caracterizacion_embeddings/resultados/original/metricas_intrinsecas.csv',
    f'{REPO}/oe4_sintesis/resultados/tabla_36_combinaciones.csv',
]
faltan = [r for r in requeridos if not os.path.exists(r)]
assert not faltan, 'Faltan archivos en el clon (haz git push de lo nuevo y vuelve a correr esta celda): ' + ', '.join(faltan)
print('OK')

## 2. Configuración

In [ ]:
import sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, REPO)
sys.path.insert(0, f'{REPO}/oe3_caracterizacion_embeddings')

import caracterizacion as car                      # extraccion de las 4 estrategias de pooling (la misma del OE3)
import transformaciones_embeddings as te           # transformaciones y metricas
from shiwilu.clasificacion import quitar_puntuacion

PRUEBA = False                                     # True: solo mBERT y el corpus original, con 5 permutaciones (comprobacion rapida)
CORPUS = ['original', 'retrotraduccion', 'generate_then_refine']
MODELOS = ['mbert', 'labse', 'xlmr']
N_PERM = 50
if PRUEBA:
    CORPUS, MODELOS, N_PERM = ['original'], ['mbert'], 5

CARPETA = '/content/drive/MyDrive/shiwilu_geometria'
os.makedirs(CARPETA, exist_ok=True)
print('Corpus:', CORPUS, '| Modelos:', MODELOS, '| Permutaciones:', N_PERM)
print('Transformaciones:', te.TRANSFORMACIONES)

## 3. Extraer los embeddings (una sola vez) y guardarlos

Se usa exactamente la extracción del OE3 (`caracterizacion.extraer_todas_las_estrategias`): CLS, mean, max y combinación de capas, sobre el texto normalizado. Si ya hay un archivo en tu Drive, no se vuelve a extraer.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no hay (irá lento)')

DATOS = {}      # corpus -> (textos normalizados, etiquetas)
EMB = {}        # (corpus, modelo) -> {pooling: matriz}
for corpus in CORPUS:
    df = car.cargar_corpus_variante(corpus)
    textos = [quitar_puntuacion(t, quitar_tildes=True) for t in df['shiwilu']]
    DATOS[corpus] = (textos, df['intencion'].to_numpy())
    for modelo in MODELOS:
        ruta = f'{CARPETA}/embeddings_{corpus}_{modelo}.npz'
        if os.path.exists(ruta):
            z = np.load(ruta); EMB[(corpus, modelo)] = {p: z[p] for p in car.ESTRATEGIAS}
            print(f'{corpus:<22} {modelo:<6} leido de Drive ({len(textos)} oraciones)')
            continue
        t0 = time.time()
        EMB[(corpus, modelo)] = car.extraer_todas_las_estrategias(textos, modelo)
        np.savez(ruta, **EMB[(corpus, modelo)])
        print(f'{corpus:<22} {modelo:<6} extraido en {time.time()-t0:.0f} s ({len(textos)} oraciones)')

## 4. Comprobación: el espacio crudo debe reproducir las cifras del OE3

Antes de transformar nada, se recalculan las métricas **crudas** y se comparan con `oe3_caracterizacion_embeddings/resultados/original/metricas_intrinsecas.csv`. Deben coincidir (diferencias de ~0.001 por la versión de las librerías). Si no coinciden, no se debe interpretar lo que sigue.

In [ ]:
previo = pd.read_csv(f'{REPO}/oe3_caracterizacion_embeddings/resultados/original/metricas_intrinsecas.csv')
if 'original' in CORPUS:
    filas = []
    for modelo in MODELOS:
        for p, X in EMB[('original', modelo)].items():
            m = car.evaluar_interseco(X, DATOS['original'][1])
            ref = previo[(previo.modelo == modelo) & (previo.estrategia == p)].iloc[0]
            filas.append({'modelo': modelo, 'pooling': p, 'silueta_ahora': m['silueta_coseno'], 'silueta_OE3': ref.silueta_coseno,
                          'dif': m['silueta_coseno'] - ref.silueta_coseno})
    chk = pd.DataFrame(filas); display(chk.round(4))
    assert chk['dif'].abs().max() < 0.005, 'NO coincide con el OE3: revisar antes de seguir'
    print('OK: el espacio crudo reproduce el OE3 (max diferencia %.5f)' % chk['dif'].abs().max())

## 5. Calcular todo

Para cada corpus, modelo y pooling: las 7 transformaciones, con silueta coseno, Davies-Bouldin, Calinski-Harabasz y la silueta con etiquetas permutadas. Más el control de n-gramas.

In [ ]:
t0 = time.time()
partes = []
for corpus in CORPUS:
    textos, y = DATOS[corpus]
    for modelo in MODELOS:
        partes.append(te.evaluar_todas({(modelo, p): X for p, X in EMB[(corpus, modelo)].items()}, y, n_perm=N_PERM, corpus=corpus))
        print(f'{corpus:<22} {modelo:<6} listo  ({time.time()-t0:.0f} s)')
    partes.append(te.control_tfidf(textos, y, n_perm=N_PERM, corpus=corpus))
R = pd.concat(partes, ignore_index=True)
R['silueta_menos_azar'] = R['silueta_coseno'] - R['silueta_azar_p97_5']      # > 0: por encima del 97.5% de lo que da el azar
R['por_encima_del_azar'] = R['silueta_coseno'] > R['silueta_azar_p97_5']
R.to_csv(f'{CARPETA}/silueta_transformaciones.csv', index=False, encoding='utf-8')
print('\nguardado en', f'{CARPETA}/silueta_transformaciones.csv', '|', len(R), 'filas')

## 6. Resultado principal: silueta (coseno) por transformación, corpus sin aumento

- Cada fila es un modelo y un pooling; cada columna, una transformación.
- La última fila es el **control de n-gramas** (sin modelo de lenguaje).
- **Importante:** la silueta del azar *no es 0* y cambia con la transformación (ver la tabla siguiente).

In [ ]:
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
ORD = te.TRANSFORMACIONES
o = R[(R.corpus == 'original') & (R.modelo != 'n-gramas (TF-IDF)')]
tabla = o.pivot_table(index=['modelo', 'pooling'], columns='transformacion', values='silueta_coseno')[ORD]
display(tabla.round(3))
ng = R[(R.corpus == 'original') & (R.modelo == 'n-gramas (TF-IDF)')].set_index('transformacion')['silueta_coseno']
print('Control n-gramas (sin modelo de lenguaje): TF-IDF crudo =', round(ng['crudo'], 3), '| TF-IDF reducido a 50 dim (SVD) =', round(ng['svd50'], 3))

### ¿Cuánto da el azar en cada espacio? (percentil 97.5 de 50 permutaciones de las etiquetas)

Una silueta solo indica agrupamiento real si **supera** este valor. Fíjate en que, en el espacio crudo, el azar ya es negativo.

In [ ]:
azar = o.pivot_table(index=['modelo', 'pooling'], columns='transformacion', values='silueta_azar_p97_5')[ORD]
display(azar.round(3))
print('Silueta MENOS azar (positivo = por encima del azar):')
display((tabla - azar).round(3))

### Cuántas de las 12 combinaciones (corpus sin aumento) quedan por encima del azar, por transformación

In [ ]:
resumen = (o.groupby('transformacion')
             .agg(silueta_media=('silueta_coseno', 'mean'), silueta_max=('silueta_coseno', 'max'),
                  por_encima_del_azar=('por_encima_del_azar', 'sum'), de=('por_encima_del_azar', 'size'))
             .loc[ORD])
display(resumen.round(3))

## 7. Davies-Bouldin (menor = mejor) y Calinski-Harabasz (mayor = mejor)

In [ ]:
print('Davies-Bouldin (menor = mejor):')
display(o.pivot_table(index=['modelo', 'pooling'], columns='transformacion', values='davies_bouldin')[ORD].round(2))
print('Calinski-Harabasz (mayor = mejor):')
display(o.pivot_table(index=['modelo', 'pooling'], columns='transformacion', values='calinski_harabasz')[ORD].round(2))
print('Nota: DB y CH se calculan sobre vectores normalizados a norma 1 (como en el OE3) y no tienen un valor de referencia fijo; se comparan dentro de una columna.')

## 8. ¿La silueta transformada anticipa mejor el F1 del clasificador?

En el OE3 la silueta cruda se relacionaba con el F1 (Spearman ≈ 0.75 en las 36 combinaciones). Aquí se mide si alguna transformación mejora esa relación. El F1 sale de `oe4_sintesis/resultados/tabla_36_combinaciones.csv` (protocolo B). Es solo una relación entre rankings; no es causal.

In [ ]:
from scipy.stats import spearmanr
F1 = pd.read_csv(f'{REPO}/oe4_sintesis/resultados/tabla_36_combinaciones.csv')
nombre_corpus = {'original': 'sin aumento', 'retrotraduccion': 'Retrotraduccion', 'generate_then_refine': 'GtR'}
M = R[R.modelo != 'n-gramas (TF-IDF)'].copy(); M['corpus_f1'] = M.corpus.map(nombre_corpus)
M = M.merge(F1[['modelo', 'pooling', 'corpus', 'f1_macro']], left_on=['modelo', 'pooling', 'corpus_f1'], right_on=['modelo', 'pooling', 'corpus'], how='inner', suffixes=('', '_f'))
filas = []
for t in ORD:
    x = M[M.transformacion == t]
    s12 = x[x.corpus == 'original']
    filas.append({'transformacion': t, 'n_todas': len(x), 'spearman_silueta_F1_todas': spearmanr(x.silueta_coseno, x.f1_macro)[0],
                  'spearman_DB_F1_todas': spearmanr(x.davies_bouldin, x.f1_macro)[0],
                  'spearman_silueta_F1_sin_aumento': spearmanr(s12.silueta_coseno, s12.f1_macro)[0]})
corr = pd.DataFrame(filas).set_index('transformacion')
corr.to_csv(f'{CARPETA}/correlacion_con_f1.csv', encoding='utf-8')
display(corr.round(3))

## 9. Figura: silueta por transformación (sin aumento), con la banda del azar

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
col = {'cls': '#1976d2', 'mean_pooling': '#2e7d32', 'max_pooling': '#f57c00', 'combinacion_capas': '#6a1b9a'}
for ax, modelo in zip(axes, MODELOS):
    sub = o[o.modelo == modelo]
    for p, g in sub.groupby('pooling'):
        g = g.set_index('transformacion').loc[ORD]
        ax.plot(range(len(ORD)), g.silueta_coseno, 'o-', color=col[p], label=p, lw=1.6)
    az = sub.groupby('transformacion').silueta_azar_p97_5.mean().loc[ORD]
    ax.plot(range(len(ORD)), az.values, '--', color='#757575', lw=1.4, label='azar (p97.5)')
    ax.axhline(0, color='k', lw=0.6)
    ax.set_xticks(range(len(ORD))); ax.set_xticklabels(ORD, rotation=40, ha='right', fontsize=8)
    ax.set_title(modelo); ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Silueta (coseno)'); axes[0].legend(fontsize=8, frameon=False)
if 'n-gramas (TF-IDF)' in set(R.modelo):
    for ax in axes: ax.axhline(ng['crudo'], color='#c62828', ls='--', lw=1)
    axes[-1].text(len(ORD) - 1, ng['crudo'], ' n-gramas', color='#c62828', va='bottom', ha='right', fontsize=8)
plt.tight_layout(); plt.savefig(f'{CARPETA}/fig_silueta_transformaciones.png', dpi=150, bbox_inches='tight'); plt.show()

## 10. Cómo leerlo (reglas fijadas de antemano)

1. **Si una transformación sube la silueta por encima del azar** (columna «Silueta menos azar» > 0) en la mayoría de las combinaciones, la silueta ≈ 0 del OE3 se debía en buena parte a la geometría del espacio (anisotropía / dimensiones dominantes), no a que los embeddings no contengan información de la intención. Eso **no cambia el F1** de los clasificadores, que ya estaban usando esa información.
2. **Si ninguna la sube por encima del azar**, la conclusión del OE3 se refuerza: los embeddings congelados no agrupan por intención ni siquiera corrigiendo su geometría.
3. **Mirar el control de n-gramas.** Si el TF-IDF de n-gramas de letras tiene una silueta mucho mayor que cualquier embedding, es coherente con que el clasificador de n-gramas supere a los modelos congelados: en este corpus la intención está en la forma de las palabras.
4. **Una transformación que sube la silueta no es un «método ganador».** Es un diagnóstico de la geometría; el resultado principal de la tesis sigue siendo el F1 de clasificación.

**Limitaciones.**
- Las transformaciones se ajustan con las mismas oraciones que se evalúan (sin usar etiquetas); en corpus con sintéticos (retrotraducción, GtR) la transformación incluye también las oraciones sintéticas.
- El blanqueo completo no es posible (menos oraciones que dimensiones); se usa PCA a 50 componentes.
- 50 permutaciones dan un percentil 97.5 aproximado del azar.
- La silueta es una medida global: un espacio puede ser perfectamente separable con un plano (lo que hace un clasificador) y tener silueta baja.

## 11. Descargar los resultados

Copia los archivos a `oe3_caracterizacion_embeddings/resultados/transformaciones/` del repositorio (revísalos antes de hacer `git add`). Los embeddings (`.npz`) quedan solo en tu Drive; **no los subas al repositorio**.

In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile('/content/geometria_resultados.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for nombre in ['silueta_transformaciones.csv', 'correlacion_con_f1.csv', 'fig_silueta_transformaciones.png']:
        assert os.path.exists(f'{CARPETA}/{nombre}'), nombre
        z.write(f'{CARPETA}/{nombre}', nombre)
files.download('/content/geometria_resultados.zip')